In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F


In [2]:
spark = SparkSession.builder \
    .appName("Olist_ETL") \
    .master("yarn") \
    .config("spark.submit.deployMode", "client") \
    .enableHiveSupport() \
    .getOrCreate()


## Creating silver Database in hive to store cleaned data

In [3]:
spark.sql("create database if not exists silver;")

2026-04-26 02:48:29,525 WARN conf.HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
2026-04-26 02:48:29,526 WARN conf.HiveConf: HiveConf of name hive.stats.retries.wait does not exist
2026-04-26 02:48:31,224 WARN metastore.ObjectStore: Failed to get database global_temp, returning NoSuchObjectException
2026-04-26 02:48:31,235 ERROR metastore.RetryingHMSHandler: AlreadyExistsException(message:Database silver already exists)
	at org.apache.hadoop.hive.metastore.HiveMetaStore$HMSHandler.create_database(HiveMetaStore.java:925)
	at sun.reflect.NativeMethodAccessorImpl.invoke0(Native Method)
	at sun.reflect.NativeMethodAccessorImpl.invoke(NativeMethodAccessorImpl.java:62)
	at sun.reflect.DelegatingMethodAccessorImpl.invoke(DelegatingMethodAccessorImpl.java:43)
	at java.lang.reflect.Method.invoke(Method.java:498)
	at org.apache.hadoop.hive.metastore.RetryingHMSHandler.invokeInternal(RetryingHMSHandler.java:148)
	at org.apache.hadoop.hive.metastore.RetryingHMSHandler.invoke(Ret

DataFrame[]

# ** Cleaning Products table **

In [4]:
p_path="/staging_zone/Products"
p_df=( spark.read
 .parquet(p_path) )

In [5]:
p_df.show()

2026-04-26 02:48:33,705 WARN parquet.CorruptStatistics: Ignoring statistics because this file was created prior to 1.8.0, see PARQUET-251


+--------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|          product_id|product_category_name|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+--------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|00066f42aeeb9f300...|           perfumaria|                 53|                       596|                 6|          300.00|            20.00|            16.00|           16.00|
|00088930e925c41fd...|           automotivo|                 56|                       752|                 4|         1225.00|            55.00|            10.00|           26.00|
|0009406fd7479715e...|      cama_mesa_banho|                 50|                       266|    

## Checking Schema

In [7]:
p_df.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: string (nullable = true)
 |-- product_length_cm: string (nullable = true)
 |-- product_height_cm: string (nullable = true)
 |-- product_width_cm: string (nullable = true)



In [8]:
p_df = (
    p_df
    # Rename columns
    .withColumnRenamed("product_name_lenght", "product_name_length")
    .withColumnRenamed("product_description_lenght", "product_description_length")

    # Cast numeric columns
    .withColumn("product_weight_g", F.col("product_weight_g").cast("double"))
    .withColumn("product_length_cm", F.col("product_length_cm").cast("double"))
    .withColumn("product_height_cm", F.col("product_height_cm").cast("double"))
    .withColumn("product_width_cm", F.col("product_width_cm").cast("double"))

    # Drop unneeded columns
    .drop(
        "product_name_length",
        "product_description_length"
    )
)

## Checking for nulls

In [9]:

p_df.select([
    F.sum(F.when(F.col(c).isNull(),1).otherwise(0)).alias(c)
    for c in p_df.columns
]).show()

+----------+---------------------+------------------+----------------+-----------------+-----------------+----------------+
|product_id|product_category_name|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+----------+---------------------+------------------+----------------+-----------------+-----------------+----------------+
|         0|                  610|               610|               2|                2|                2|               2|
+----------+---------------------+------------------+----------------+-----------------+-----------------+----------------+



## Dropping NULL values

In [10]:
products_clean = p_df.dropna(subset=[
    "product_category_name",
    "product_photos_qty"
])

## Filling NULL Dimensions 

In [13]:
products_clean = products_clean.fillna({
    "product_weight_g":0,
    "product_length_cm":0,
    "product_height_cm":0,
    "product_width_cm":0
})

In [14]:
products_clean.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: double (nullable = false)
 |-- product_length_cm: double (nullable = false)
 |-- product_height_cm: double (nullable = false)
 |-- product_width_cm: double (nullable = false)



## Save cleaned products as external table in hive

In [15]:
p_cleaned_path="/silver/products"
(products_clean
.write
.mode("overwrite")
.option("path",p_cleaned_path) 
.saveAsTable("silver.products"))

2026-04-26 02:49:22,175 WARN session.SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.
2026-04-26 02:49:22,231 WARN conf.HiveConf: HiveConf of name hive.internal.ss.authz.settings.applied.marker does not exist
2026-04-26 02:49:22,231 WARN conf.HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
2026-04-26 02:49:22,232 WARN conf.HiveConf: HiveConf of name hive.stats.retries.wait does not exist


# ** Cleaning Customers Table **

In [16]:
c_path="/staging_zone/Customers"
c_df=( spark.read
 .parquet(c_path) )

In [17]:
c_df.show()

+--------------------+--------------------+------------------------+--------------------+--------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|       customer_city|customer_state|
+--------------------+--------------------+------------------------+--------------------+--------------+
|00012a2ce6f8dcda2...|248ffe10d632bebe4...|                   06273|              osasco|            SP|
|000161a058600d590...|b0015e09bb4b6e47c...|                   35550|         itapecerica|            MG|
|0001fd6190edaaf88...|94b11d37cd61cb299...|                   29830|        nova venecia|            ES|
|0002414f953443074...|4893ad4ea28b2c5b3...|                   39664|            mendonca|            MG|
|000379cdec6255224...|0b83f73b19c2019e1...|                   04841|           sao paulo|            SP|
|0004164d20a9e969a...|104bdb7e6a6cdceaa...|                   13272|            valinhos|            SP|
|000419c5494106c30...|14843983d4a159080...|            

## Checking for NULLS

In [18]:
c_df.select([
    F.sum(F.when(F.col(c).isNull(),1).otherwise(0)).alias(c)
    for c in c_df.columns
]).show()

+-----------+------------------+------------------------+-------------+--------------+
|customer_id|customer_unique_id|customer_zip_code_prefix|customer_city|customer_state|
+-----------+------------------+------------------------+-------------+--------------+
|          0|                 0|                       0|            0|             0|
+-----------+------------------+------------------------+-------------+--------------+



## Checking for Duplicates

In [19]:
c_df.groupBy("customer_id").count().filter("count > 1").show()

+-----------+-----+
|customer_id|count|
+-----------+-----+
+-----------+-----+



In [20]:
c_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)



## Cleaned Customers 

In [21]:
from pyspark.sql.functions import col, trim, lower, upper, initcap, regexp_replace, lpad

customers_clean = (
    c_df
    
    # trim strings
    .withColumn("customer_id", trim(col("customer_id")))
    .withColumn("customer_unique_id", trim(col("customer_unique_id")))
    .withColumn("customer_city", trim(col("customer_city")))
    .withColumn("customer_state", trim(col("customer_state")))

    # standardize text
    .withColumn("customer_city", initcap(lower(col("customer_city"))))
    .withColumn("customer_state", upper(col("customer_state")))

)


## Save cleaned cutomers as external table in hive

In [22]:
c_cleaned_path="/silver/customers"
( customers_clean
 .write
 .mode("overwrite")
 .option("path",c_cleaned_path)
 .saveAsTable("silver.customers") )

# ** Cleaning Orders table **

In [23]:
o_path="/staging_zone/Orders"
o_df=( spark.read
 .parquet(o_path) )

In [24]:
o_df.show()

+--------------------+--------------------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|00010242fe8c5a6d1...|3ce436f183e68e078...|   delivered|           1505260742000|    1505263535000|               1505813656000|                1505918628000|                1506610800000|
|00018f77f2f0320c5...|f6dd3ec061db4e398...|   delivered|           1493171586000|    1493172313000|               1493876100000|                1494572664000|                1494774000000|
|000229ec398224ef6...|6489ae5e4333f3693...|   delivered

## Duplicates check

In [25]:
o_df.groupBy("order_id").count().filter("count > 1").show()

+--------+-----+
|order_id|count|
+--------+-----+
+--------+-----+



## Check for NULLS

In [26]:
o_df.select([
    F.sum(F.when(F.col(c).isNull(),1).otherwise(0)).alias(c)
    for c in o_df.columns
]).show()

+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|order_id|customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|       0|          0|           0|                       0|              160|                        1783|                         2965|                            0|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+



## Cleaned orders

In [27]:
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col, trim, lower, to_timestamp,
    from_unixtime, datediff
)

orders_clean = (
    o_df
    # clean strings`
    .withColumn("order_id", trim(col("order_id")))
    .withColumn("customer_id", trim(col("customer_id")))
    .withColumn("order_status", lower(trim(col("order_status"))))

    # epoch ms -> timestamp
    .withColumn(
        "order_purchase_timestamp",
        to_timestamp(from_unixtime(col("order_purchase_timestamp") / 1000))
    )
    .withColumn(
        "order_approved_at",
        to_timestamp(from_unixtime(col("order_approved_at") / 1000))
    )
    .withColumn(
        "order_delivered_carrier_date",
        to_timestamp(from_unixtime(col("order_delivered_carrier_date") / 1000))
    )
    .withColumn(
        "order_delivered_customer_date",
        to_timestamp(from_unixtime(col("order_delivered_customer_date") / 1000))
    )
    .withColumn(
        "order_estimated_delivery_date",
        to_timestamp(from_unixtime(col("order_estimated_delivery_date") / 1000))
    )

    # flags
    .withColumn("is_approved", F.col("order_approved_at").isNotNull())
    .withColumn("is_shipped", F.col("order_delivered_carrier_date").isNotNull())
    .withColumn("is_delivered", F.col("order_delivered_customer_date").isNotNull())

    # metrics
     .withColumn(
        "delivery_days",
        F.when(
            F.col("order_delivered_customer_date").isNotNull(),
            F.datediff(
                F.col("order_delivered_customer_date"),
                F.col("order_purchase_timestamp")
            )
        ).otherwise(F.lit(None)) )
    # is_late
    .withColumn(
        "is_late",
        F.when(
            F.col("order_delivered_customer_date").isNotNull() &
            F.col("order_estimated_delivery_date").isNotNull(),
            F.col("order_delivered_customer_date") >
            F.col("order_estimated_delivery_date")
        ).otherwise(F.lit(None).cast("boolean"))
    )
)
orders_clean.show(1)


+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+-----------+----------+------------+-------------+-------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|is_approved|is_shipped|is_delivered|delivery_days|is_late|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+-----------+----------+------------+-------------+-------+
|00010242fe8c5a6d1...|3ce436f183e68e078...|   delivered|     2017-09-13 08:59:02|2017-09-13 09:45:35|         2017-09-19 18:34:16|          2017-09-20 23:43:48|          2017-09-29 00:00:00|       true|      true|        true|            7|  false|
+---

## Save cleaned orders as external table in hive

In [28]:
orders_path="/silver/orders"
(
 orders_clean
 .write
 .mode("overwrite")
 .option("path",orders_path)
 .saveAsTable("silver.orders")   
)

# ** Cleaning Order_Items **

In [29]:
o_i_path="/staging_zone/Order_Items"
o_i_df=( spark.read
 .parquet(o_i_path) )

In [30]:
o_i_df.show()

+--------------------+-------------+--------------------+--------------------+-------------------+------+-------------+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_date| price|freight_value|
+--------------------+-------------+--------------------+--------------------+-------------------+------+-------------+
|00010242fe8c5a6d1...|            1|4244733e06e7ecb49...|48436dade18ac8b2b...|      1505781935000| 58.90|        13.29|
|00018f77f2f0320c5...|            1|e5f2d52b802189ee6...|dd7ddc04e1b6c2c61...|      1493777113000|239.90|        19.93|
|000229ec398224ef6...|            1|c777355d18b72b67a...|5b51032eddd242adc...|      1516254510000|199.00|        17.87|
|00024acbcdf0a6daa...|            1|7634da152a4610f15...|9d7a1d34a50524090...|      1534295418000| 12.99|        12.79|
|00042b26cf59d7ce6...|            1|ac6c3623068f30de0...|df560393f3a51e745...|      1486961871000|199.90|        18.14|
|00048cc3ae777c65d...|            1|ef92

## Checking Schema

In [31]:
o_i_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: long (nullable = true)
 |-- price: string (nullable = true)
 |-- freight_value: string (nullable = true)



## Checking for NULLS

In [32]:
o_i_df.select([
    F.sum(F.when(F.col(c).isNull(),1).otherwise(0)).alias(c)
    for c in o_i_df.columns
]).show()

+--------+-------------+----------+---------+-------------------+-----+-------------+
|order_id|order_item_id|product_id|seller_id|shipping_limit_date|price|freight_value|
+--------+-------------+----------+---------+-------------------+-----+-------------+
|       0|            0|         0|        0|                  0|    0|            0|
+--------+-------------+----------+---------+-------------------+-----+-------------+



## Checking for duplicates

In [33]:
o_i_df \
    .groupBy("order_id", "order_item_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()

+--------+-------------+-----+
|order_id|order_item_id|count|
+--------+-------------+-----+
+--------+-------------+-----+



## Cleaned orders_items

In [34]:
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col, trim, to_timestamp, from_unixtime, round
)

order_items_clean = (
    o_i_df
    # clean ids
    .withColumn("order_id", trim(col("order_id")))
    .withColumn("product_id", trim(col("product_id")))
    .withColumn("seller_id", trim(col("seller_id")))

    # proper numeric types
    .withColumn("order_item_id", col("order_item_id").cast("int"))
    .withColumn("price", col("price").cast("double"))
    .withColumn("freight_value", col("freight_value").cast("double"))

    # epoch ms -> timestamp
    .withColumn(
        "shipping_limit_date",
        to_timestamp(from_unixtime(col("shipping_limit_date") / 1000))
    )

    # derived metrics
    .withColumn(
        "is_free_shipping",
        (col("freight_value") == 0)
    )
)

order_items_clean.show(1)

+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+----------------+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_date|price|freight_value|is_free_shipping|
+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+----------------+
|00010242fe8c5a6d1...|            1|4244733e06e7ecb49...|48436dade18ac8b2b...|2017-09-19 09:45:35| 58.9|        13.29|           false|
+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+----------------+
only showing top 1 row



## Save cleaned order_items as external table in hive

In [35]:
order_items_path="/silver/order_items"
(
 order_items_clean
 .write
 .mode("overwrite")
 .option("path",order_items_path)
 .saveAsTable("silver.order_items")   
)

# ** Cleaning Order_Reviews **

In [36]:
o_r_path="/staging_zone/Order_Reviews"
o_r_df=( spark.read
 .parquet(o_r_path) )

In [37]:
o_r_df.show()

+--------------------+--------------------+------------+--------------------+----------------------+--------------------+-----------------------+
|           review_id|            order_id|review_score|review_comment_title|review_comment_message|review_creation_date|review_answer_timestamp|
+--------------------+--------------------+------------+--------------------+----------------------+--------------------+-----------------------+
|0001239bc1de2e33c...|fc046d77761718714...|           5|                null|                  null|       1521471600000|          1521538564000|
|0001cc6860aeaf5b9...|d4665434b01caa9dc...|           5|                null|                  null|       1529593200000|          1529988689000|
|00020c7512a52e922...|e28abf2eb2f1fbcbd...|           5|    Entrega rÃ¡pida!|  A entrega foi sup...|       1524582000000|          1524722136000|
|00032b0141443497c...|04fb47576993a3cb0...|           5|                null|                  null|       1496070000000|   

## Checking Schema

In [38]:
o_r_df.printSchema()

root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: integer (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: long (nullable = true)
 |-- review_answer_timestamp: long (nullable = true)



## Cleaned Order_Reviews

In [39]:
from pyspark.sql.functions import (
    col, trim, length,
    to_timestamp, from_unixtime, datediff
)

order_reviews_clean = (
    o_r_df
    # clean ids
    .withColumn("review_id", trim(col("review_id")))
    .withColumn("order_id", trim(col("order_id")))
    
    # derived columns
    .withColumn(
        "has_title",
        col("review_comment_title").isNotNull()
    )
    .withColumn(
        "has_message",
        col("review_comment_message").isNotNull()
    )
    
    # convert timestamps
    .withColumn(
        "review_creation_date",
        to_timestamp(from_unixtime(col("review_creation_date") / 1000))
    )
    .withColumn(
        "review_answer_timestamp",
        to_timestamp(from_unixtime(col("review_answer_timestamp") / 1000))
    )


    # drop raw review message assuming data onlu used in analytics no ml
    .drop("review_comment_message")
    .drop("review_comment_title")
)

order_reviews_clean.show()


+--------------------+--------------------+------------+--------------------+-----------------------+---------+-----------+
|           review_id|            order_id|review_score|review_creation_date|review_answer_timestamp|has_title|has_message|
+--------------------+--------------------+------------+--------------------+-----------------------+---------+-----------+
|0001239bc1de2e33c...|fc046d77761718714...|           5| 2018-03-20 00:00:00|    2018-03-20 18:36:04|    false|      false|
|0001cc6860aeaf5b9...|d4665434b01caa9dc...|           5| 2018-06-22 00:00:00|    2018-06-26 13:51:29|    false|      false|
|00020c7512a52e922...|e28abf2eb2f1fbcbd...|           5| 2018-04-25 00:00:00|    2018-04-26 14:55:36|     true|       true|
|00032b0141443497c...|04fb47576993a3cb0...|           5| 2017-05-30 00:00:00|    2017-06-01 23:28:55|    false|      false|
|00034d88989f9a4c3...|5f358d797a49fe2f2...|           5| 2017-08-12 00:00:00|    2017-08-13 19:56:53|    false|      false|
|000359b

## Checking for NULLS

In [40]:
order_reviews_clean.select([
    F.sum(F.when(F.col(c).isNull(),1).otherwise(0)).alias(c)
    for c in order_reviews_clean.columns
]).show()

+---------+--------+------------+--------------------+-----------------------+---------+-----------+
|review_id|order_id|review_score|review_creation_date|review_answer_timestamp|has_title|has_message|
+---------+--------+------------+--------------------+-----------------------+---------+-----------+
|        0|       0|           0|                   1|                      1|        0|          0|
+---------+--------+------------+--------------------+-----------------------+---------+-----------+



## Save cleaned order_reviews as external table in hive

In [41]:
order_items_path="/silver/order_reviews"
(
 order_reviews_clean
 .write
 .mode("overwrite")
 .option("path",order_items_path)
 .saveAsTable("silver.order_reviews")   
)